# Kazhdan–Lusztig data of the tetrahedron example (review version)

**Quantum polynomials ($\epsilon$ in terms of $\nu$), block inversion ($\nu$ in terms of $\epsilon$) and characters.**

This notebook starts from the data already extracted from the main notebook (file `test_review.g`), so it does **not**
run the main loop. It is meant to be executed **once, from top to bottom** (*Run All*) with the GAP kernel.

**How to run**
1. Edit only the two lines of cell 0.1 (path of `test_review.g` and output directory).
2. *Kernel → Restart & Run All*.
3. At the end, section 7 prints `N of N checks passed`. Every check line must start with `[ OK ]`.

**Contents**

| Section | What it does |
|---|---|
| 0 | Configuration, loading of the data, sanity checks |
| 1 | Weights: $\lambda\mapsto\lambda^\vee$, $\tau$, $p$ (called `sigma`) |
| 2 | Verma characters: $\epsilon$ in terms of $\nu$ (quantum polynomials) |
| 3 | Blocks (relation with the **sum** $t_{p,q}(1)+t_{q,p}(1)\neq 0$) and coefficient matrix |
| 4 | Exact inversion by blocks: $\nu$ in terms of $\epsilon$ |
| 5 | LaTeX files; polynomialsKL and seriesKL in the format of the paper |
| 6 | Characters in the format of the paper (filtered and complete) |
| 7 | Summary of the checks |
| A, B | Annex: functions under development, optional console views |

**Notation.** `eps_p = sum_q t_{p,q}(q) nu_q` is the decomposition of the Verma character $\epsilon_p$; the blocks of the matrix
$(t_{p,q})$ are the blocks of $\operatorname{Rep}\mathfrak D$. Weights are numbered as in the enumeration of Section 5 of the paper.

## 0. Configuration and data

`TEST_REVIEW_G` is the file with the extracted data (`data_record_data_decomp`, `data_graded`, `SimpleNames`, ...).
`OUTDIR` is the directory where the `.tex` files are written (`""` = current directory; otherwise it must end with `/`).

In [2]:
# ---- Configuration: edit only these two lines -----------------------------------------------------
TEST_REVIEW_G := "/home/juandavid/Descargas/gap-4.13.1/pkg/qgnag/examples/TETRAHEDRON/KAZHDANLUSZTIG/test_review.g";;
OUTDIR        := "";;

In [3]:
Read(TEST_REVIEW_G);;

In [6]:
# Helper used by all the checks of this notebook: prints [ OK ] / [FAIL] and records the result.
NB_RESULTS := [];;
# true if the file contains the given text.
NBContains := function(filename, text)
    return PositionSublist(StringFile(filename), text) <> fail;
end;;
NBCheck := function(label, condition)
    local ok;
    ok := (condition = true);
    Add(NB_RESULTS, ok);
    if ok then
        Print("[ OK ]  ", label, "\n");
    else
        Print("[FAIL]  ", label, "\n");
    fi;
end;;

In [9]:
# Sanity checks on the loaded data.
NBCheck("56 decompositions: data_record_data_decomp", Length(data_record_data_decomp) = 56);
NBCheck("56 simple names: SimpleNames",              Length(SimpleNames) = 56);
NBCheck("56 graded Verma data: data_graded",          Length(data_graded) = 56);

[ OK ]  56 decompositions: data_record_data_decomp
[ OK ]  56 simple names: SimpleNames
[ OK ]  56 graded Verma data: data_graded


## 1. Weights and permutations

* `vee` is $\lambda\mapsto\lambda^\vee$ (the bottom weight of the socle).
* `tau` is $\tau$ (`QGNAG_PermutationVarChars`); it indexes $L(\lambda)$ by its top weight.
* `sigma` is the permutation $p$ of Lemma 6.4 (bottom weight of $L(\lambda)$).

The expected values are those of the section *Weights permutation* of the paper, and they satisfy $\tau=\lambda\mapsto p(\lambda^\vee)$.

In [12]:
data_rec_info := List(data_record_data_decomp, QGNAG_Decomposition);;

# Index of the weight at the highest degree of a decomposition.
IndexVarChars := function(rec_info)
    local degrees, max_degree, pair;
    degrees    := List(RecNames(rec_info), Int);
    max_degree := Maximum(degrees);
    pair       := rec_info.(max_degree);
    return pair[1][2];
end;;

vee := PermList(List(data_rec_info, IndexVarChars));;

In [16]:
tau               := QGNAG_PermutationVarChars(data_record_data_decomp);;
bijs_by_degree    := List(Permuted(data_record_data_decomp, tau), QGNAG_GetShiftedBijs);;
data_rec_info_neg := List(bijs_by_degree, QGNAG_MultiplicitiesToDecomposition);;
sigma             := QGNAG_PermutationFromRecInfo(data_rec_info_neg);;

In [19]:
# Decompositions with degrees <= 0 (not permuted): the highest degree must be 0 and give tau again.
data_rec_info_neg_raw := List(List(data_record_data_decomp, QGNAG_DecompositionFromNegativeDegreeToZero),
                              QGNAG_MultiplicitiesToDecomposition);;
NBCheck("top degree of every decomposition is 0",
        ForAll(data_rec_info_neg_raw, r -> Maximum(List(RecNames(r), Int)) = 0));
NBCheck("tau = PermList(top weights)", tau = PermList(List(data_rec_info_neg_raw, IndexVarChars)));

[ OK ]  top degree of every decomposition is 0
[ OK ]  tau = PermList(top weights)


In [26]:
# Expected permutations (Section "Weights permutation" of the paper).
vee_expected   := (1,10)(2,11)(3,12)(4,7)(5,8)(6,9)(13,16)(14,15)(17,26)(18,25)(19,28)(20,27)(21,30)(22,29)(23,32)(24,31)(33,48)(34,49)(35,50)(36,45)(37,46)(38,47)(39,54)(40,55)(41,56)(42,51)(43,52)(44,53);;
tau_expected   := (1,10)(2,6)(3,8)(5,12)(9,11)(13,16)(17,26)(19,28)(21,32)(22,24)(23,30)(29,31)(33,51)(34,41)(35,55)(36,39)(37,53)(38,43)(40,50)(42,48)(44,46)(45,54)(47,52)(49,56);;
sigma_expected := (2,9)(3,5)(4,7)(6,11)(8,12)(14,15)(18,25)(20,27)(21,23)(22,31)(24,29)(30,32)(33,42)(34,56)(35,40)(36,54)(37,44)(38,52)(39,45)(41,49)(43,47)(46,53)(48,51)(50,55);;

NBCheck("lambda -> lambda^vee  as in the paper", vee   = vee_expected);
NBCheck("tau                   as in the paper", tau   = tau_expected);
NBCheck("p (sigma)             as in the paper", sigma = sigma_expected);
NBCheck("tau = lambda -> p(lambda^vee)",         tau   = vee * sigma);

[ OK ]  lambda -> lambda^vee  as in the paper
[ OK ]  tau                   as in the paper
[ OK ]  p (sigma)             as in the paper
[ OK ]  tau = lambda -> p(lambda^vee)


In [27]:
# Informational (not counted): does p preserve the fusion rules?  Expected: no failures.
if IsBound(SimplesMn) and IsBound(data_fusion_rules_parallel) then
    fusion_failures := QGNAG_PermutationFusionRulesFailures(sigma, SimplesMn, data_fusion_rules_parallel);;
    Print("fusion rules failures of p: ", fusion_failures, "   (expected: an empty list)\n");
else
    Print("[SKIP]  fusion rules of p: SimplesMn / data_fusion_rules_parallel are not loaded\n");
fi;

[SKIP]  fusion rules of p: SimplesMn / data_fusion_rules_parallel are not loaded


## 2. Verma characters: $\epsilon$ in terms of $\nu$ (quantum polynomials)

Each Verma character is written as $\epsilon_p=\sum_q t_{p,q}\,\nu_q$ with $t_{p,q}\in\mathbb Z[q^{-1}]$.

* `shifted_vectors`: the shifted multiplicity vectors $\boldsymbol q^{-j}\nu_k$.
* `epsilon_and_coeffs`: Verma characters and their coefficients in that system.
* `NuVectorNamesPermuted`: LaTeX names of the shifted vectors (the identifier keeps its name from the previous notebook).
* `data_quantum_pols`: the polynomials $t_{p,q}$ as coefficient lists in $q^{-1}$; `data_quantum_pols_eval`: their values $t_{p,q}(1)$.

In [30]:
shifted_vectors       := QGNAG_AllShiftLeft(data_record_data_decomp);;
epsilon_and_coeffs    := QGNAG_VermaModuleCharactersAndCoefficients(data_graded, shifted_vectors);;
NuVectorNamesPermuted := QGNAG_NamesOfShiftedVectors(data_record_data_decomp);;

In [33]:
NBCheck("56 Verma characters", Length(epsilon_and_coeffs) = 56);
NBCheck("eps_1  = nu_1 + q^-9 nu_10 + q^-1 nu_36 + q^-4 nu_45",
        QGNAG_EpsilonToLaTeX(1, epsilon_and_coeffs, NuVectorNamesPermuted)
        = "\\epsilon_{1} = \\nu_{1} + \\boldsymbol{q}^{-9}\\nu_{10} + \\boldsymbol{q}^{-1}\\nu_{36} + \\boldsymbol{q}^{-4}\\nu_{45}");
NBCheck("eps_43 = q^-5 nu_38 + nu_43",
        QGNAG_EpsilonToLaTeX(43, epsilon_and_coeffs, NuVectorNamesPermuted)
        = "\\epsilon_{43} = \\boldsymbol{q}^{-5}\\nu_{38} + \\nu_{43}");

[ OK ]  56 Verma characters
[ OK ]  eps_1  = nu_1 + q^-9 nu_10 + q^-1 nu_36 + q^-4 nu_45
[ OK ]  eps_43 = q^-5 nu_38 + nu_43


In [34]:
# Prints the 56 relations  eps_p = sum_q t_{p,q} nu_q  (LaTeX).
QGNAG_PrintQuantumPolynomials := function(epsilon_and_coeffs, nu_names)
    local theta, index_s;
    theta := Length(epsilon_and_coeffs);
    for index_s in [1 .. theta] do
        Print(QGNAG_EpsilonToLaTeX(index_s, epsilon_and_coeffs, nu_names), "\n");
    od;
end;;

In [35]:
QGNAG_PrintQuantumPolynomials(epsilon_and_coeffs, NuVectorNamesPermuted);

\epsilon_{1} = \nu_{1} + \boldsymbol{q}^{-9}\nu_{10} + \boldsymbol{q}^{-1}\nu_{36} + \boldsymbol{q}^{-4}\nu_{45}
\epsilon_{2} = \nu_{2} + \boldsymbol{q}^{-6}\nu_{6} + \boldsymbol{q}^{-2}\nu_{42}
\epsilon_{3} = \nu_{3} + \boldsymbol{q}^{-3}\nu_{8}
\epsilon_{4} = \nu_{4}
\epsilon_{5} = \nu_{5} + \boldsymbol{q}^{-3}\nu_{12}
\epsilon_{6} = \boldsymbol{q}^{-6}\nu_{2} + \nu_{6} + \boldsymbol{q}^{-2}\nu_{42}
\epsilon_{7} = \nu_{7}
\epsilon_{8} = \boldsymbol{q}^{-3}\nu_{3} + \nu_{8}
\epsilon_{9} = \nu_{9} + \boldsymbol{q}^{-6}\nu_{11} + \boldsymbol{q}^{-2}\nu_{51}
\epsilon_{10} = \boldsymbol{q}^{-9}\nu_{1} + \nu_{10} + \boldsymbol{q}^{-4}\nu_{36} + \boldsymbol{q}^{-1}\nu_{45}
\epsilon_{11} = \boldsymbol{q}^{-6}\nu_{9} + \nu_{11} + \boldsymbol{q}^{-2}\nu_{51}
\epsilon_{12} = \boldsymbol{q}^{-3}\nu_{5} + \nu_{12}
\epsilon_{13} = \boldsymbol{q}^{-6}\nu_{1} + \boldsymbol{q}^{-3}\nu_{10} + \nu_{13} + \boldsymbol{q}^{-3}\nu_{16} + \boldsymbol{q}^{-1}\nu_{36} + \boldsymbol{q}^{-2}\nu_{39} + \boldsymb

In [37]:
data_quantum_pols      := QGNAG_QuantumPolynomialsToRecord(Permuted(data_record_data_decomp, sigma), epsilon_and_coeffs);;
data_quantum_pols_eval := QGNAG_EvaluateQuantumPolynomialsRecord(data_quantum_pols);;

Length(nu) = 248
Shift counts = [ 10, 7, 4, 1, 4, 7, 1, 4, 7, 10, 7, 4, 4, 1, 1, 4, 4, 1, 4, 1, 4, 7, 4, 7, 1, 4, 1, 4, 7, 4, 7, 4, 3, 6, 3, 6, 3, 6, 8, 5, 2, 5, 2, 5, 6, 3, 6, 3, 6, 3, 5, 2, 5, 8, 5, 2 ]


In [39]:
NBCheck("t_{p,p}(1) > 0 for every p (so that p belongs to R_p)",
        ForAll([1 .. 56], p -> data_quantum_pols_eval[p].(String(p)) > 0));
NBCheck("192 nonzero entries t_{p,q}(1)",
        Sum(List(data_quantum_pols_eval, r -> Number(RecNames(r), k -> r.(k) <> 0))) = 192);

[ OK ]  t_{p,p}(1) > 0 for every p (so that p belongs to R_p)
[ OK ]  192 nonzero entries t_{p,q}(1)


## 3. Blocks and coefficient matrix

**Blocks of $\operatorname{Rep}\mathfrak D$ (Section 4.4 of the paper).** For each $p$ let
$R_p=\{q : t_{p,q}(1)+t_{q,p}(1)\neq 0\}$ (note: the **sum**, i.e. *at least one* is nonzero; a product would require *both*).
$B_{\ell,1}=R_p$, $B_{\ell,k+1}=\bigcup_{q\in B_{\ell,k}}R_q$, with $p$ the smallest index not yet in a block.

`QGNAG_ClassifySimplesIntoBlocksByRelation` implements it. On these data the sum and the product give the same blocks, but only the sum
is the definition of the paper. `QGNAG_GraphFromQuantumPols` is symmetrized for the same reason (the support of $t$ is not symmetric).

In [40]:
#############################################################################
##
##  QGNAG_ClassifySimplesIntoBlocksByRelation( data_pols_decomp )
##
##  Implements the construction of the blocks of Section 4.4.
##
##  For each simple p, let
##      R_p = { q | t_{p,q}(1) + t_{q,p}(1) <> 0 },
##  where t_{p,q} is the polynomial data_pols_decomp[p].(String(q)) and
##  t_{p,q}(1) is its value at 1 (the sum of its coefficients).
##  Observe that p is in R_p for all p.
##
##  The l-th block is built as follows:
##      p       = min{ p | p not in B_k, k < l }
##      B_{l,1} = R_p
##      B_{l,k+1} = union of R_q over q in B_{l,k},   k >= 1
##      B_l     = B_{l,k}  for the first k such that  B_{l,k+1} = B_{l,k}
##
##  Returns a record with fields "1", "2", ... : the blocks, in the order
##  in which they are constructed (increasing minimum element).
##
##  Note: the condition is the SUM t_{p,q}(1) + t_{q,p}(1) <> 0, i.e. at least one of
##  them is nonzero (all the entries are nonnegative); it is NOT the product.
##
#############################################################################

QGNAG_ClassifySimplesIntoBlocksByRelation := function( data_pols_decomp )
    local numSimples,
          R,
          p,
          q,
          polynomial_pq,
          polynomial_qp,
          visited,
          blocks,
          blockIndex,
          block_k,
          block_next,
          stable;

    numSimples := Length( data_pols_decomp );

    # R[p] = R_p = { q | t_{p,q}(1) + t_{q,p}(1) <> 0 }.
    # The diagonal is NOT skipped: the construction needs p in R_p.
    R := List( [ 1 .. numSimples ], x -> [] );
    for p in [ 1 .. numSimples ] do
        for q in [ 1 .. numSimples ] do
            polynomial_pq := data_pols_decomp[p].(String(q));
            polynomial_qp := data_pols_decomp[q].(String(p));
            # Sum of the coefficients = value of the polynomial at 1
            if Sum( polynomial_pq ) + Sum( polynomial_qp ) <> 0 then
                Add( R[p], q );
            fi;
        od;
        # The paper observes that p is in R_p for all p; if this fails the
        # iteration below would not terminate correctly, so stop here.
        if not p in R[p] then
            Error( "QGNAG_ClassifySimplesIntoBlocksByRelation: p = ", p,
                   " is not in R_p (t_{p,p}(1) = 0)" );
        fi;
    od;

    # visited[p] = true  iff p already belongs to some block B_k, k < l.
    visited    := BlistList( [ 1 .. numSimples ], [] );
    blocks     := rec();
    blockIndex := 1;

    for p in [ 1 .. numSimples ] do
        # Scanning in increasing order, the first unvisited p is
        # p = min{ p | p not in B_k, k < l }.
        if not visited[p] then
            # B_{l,1} = R_p
            block_k := Set( R[p] );
            repeat
                # B_{l,k+1} = union of R_q over q in B_{l,k}
                block_next := Set( Concatenation( List( block_k, x -> R[x] ) ) );
                stable     := ( block_next = block_k );
                block_k    := block_next;
            until stable;

            # B_l = B_{l,k}
            for q in block_k do
                visited[q] := true;
            od;
            blocks.(String( blockIndex )) := block_k;
            blockIndex := blockIndex + 1;
        fi;
    od;

    return blocks;
end;

function( data_pols_decomp ) ... end

In [40]:
# Graph of the blocks: an edge [p,q], p < q, whenever t_{p,q}(1) + t_{q,p}(1) <> 0.
QGNAG_GraphFromQuantumPols := function(data_quantum_pols_eval)
    local theta, edges, p, q;
    theta := Length(data_quantum_pols_eval);
    edges := [];
    for p in [1 .. theta] do
        for q in [p + 1 .. theta] do
            if data_quantum_pols_eval[p].(String(q)) + data_quantum_pols_eval[q].(String(p)) <> 0 then
                Add(edges, [p, q]);
            fi;
        od;
    od;
    return rec(nodes := [1 .. theta], edges := edges);
end;;

In [47]:
block_classification := QGNAG_ClassifySimplesIntoBlocksByRelation(data_quantum_pols);;
block_graph          := QGNAG_GraphFromQuantumPols(data_quantum_pols_eval);;

NBCheck("23 blocks",
        Length(RecNames(block_classification)) = 23);
NBCheck("block sizes: 8 of size 1, 12 of size 2, 2 of size 6, 1 of size 12",
        Collected(List(RecNames(block_classification), n -> Length(block_classification.(n)))) = [[1,8],[2,12],[6,2],[12,1]]);
NBCheck("first block = {1,10,13,16,17,19,26,28,36,39,45,54}",
        block_classification.("1") = [1,10,13,16,17,19,26,28,36,39,45,54]);
NBCheck("block graph: 87 edges", Length(block_graph.edges) = 87);

[ OK ]  23 blocks
[ OK ]  block sizes: 8 of size 1, 12 of size 2, 2 of size 6, 1 of size 12
[ OK ]  first block = {1,10,13,16,17,19,26,28,36,39,45,54}
[ OK ]  block graph: 87 edges


**Coefficient matrix $M$ (with $\epsilon=M\nu$) and its block decomposition.**
`QGNAG_TestCoefficientMatrix` returns the list of failures (expected: empty).

In [51]:
nu_coeffs_pkg := QGNAG_QuamtumCoefficientsShifted(data_record_data_decomp);;
base          := QGNAG_BaseIndexOfColumns(nu_coeffs_pkg);;
M             := QGNAG_CoefficientMatrix(epsilon_and_coeffs, nu_coeffs_pkg, base);;
blocks        := QGNAG_BlockDecomposition(M);;

In [56]:
NBCheck("coefficient matrix reproduces the Verma characters", QGNAG_TestCoefficientMatrix(epsilon_and_coeffs, nu_coeffs_pkg, base, M) = []);
NBCheck("every block is square (rows = columns)",              ForAll(blocks, b -> b.rows = b.cols));
NBCheck("blocks of M: sizes 8 x 1, 12 x 2, 2 x 6, 1 x 12",     Collected(List(blocks, b -> Length(b.rows))) = [[1,8],[2,12],[6,2],[12,1]]);
NBCheck("first block of M = {1,10,13,16,17,19,26,28,36,39,45,54}", blocks[1].cols = [1,10,13,16,17,19,26,28,36,39,45,54]);
NBCheck("classification (sum) = block decomposition of M",
        Set(List(RecNames(block_classification), n -> block_classification.(n))) = Set(List(blocks, b -> Set(b.cols))));

[ OK ]  coefficient matrix reproduces the Verma characters
[ OK ]  every block is square (rows = columns)
[ OK ]  blocks of M: sizes 8 x 1, 12 x 2, 2 x 6, 1 x 12
[ OK ]  first block of M = {1,10,13,16,17,19,26,28,36,39,45,54}
[ OK ]  classification (sum) = block decomposition of M


In [60]:
# The block-diagonal matrix and the blocks reproduce  eps = M nu  (nus are indeterminates).
full := QGNAG_BlockDiagonalMatrix(M);;
nus  := List([1 .. Length(data_record_data_decomp)], k -> Indeterminate(Rationals, Concatenation("\\nu_{", String(k), "}")));;
NBCheck("block-diagonal matrix: eps = M nu",
        ForAll([1 .. Length(data_record_data_decomp)], r -> full.matrix[r] * nus{full.cols} = M[full.rows[r]] * nus));
NBCheck("each block: eps = M nu",
        ForAll(blocks, b -> ForAll([1 .. Length(b.rows)], r -> b.matrix[r] * nus{b.cols} = M[b.rows[r]] * nus)));

[ OK ]  block-diagonal matrix: eps = M nu
[ OK ]  each block: eps = M nu


## 4. Exact inversion by blocks: $\nu$ in terms of $\epsilon$

`QGNAG_InvertBlocks` inverts every block over $\mathbb Q(q)$ and stores the determinant. For a block of size $2k$ the inverse is
$\mathfrak d^{k}\times$(Laurent polynomial in $q^{-1}$), with $\mathfrak d=(1-q^{-6})^{-1}$; this is the compressed form used in the paper.

In [61]:
inv_blocks := QGNAG_InvertBlocks(blocks);;

In [64]:
NBCheck("QGNAG_TestInverseBlocks",  QGNAG_TestInverseBlocks(inv_blocks) = true);
NBCheck("inverse * matrix = identity in every block",
        ForAll(inv_blocks, b -> b.inverse * b.matrix = IdentityMat(Length(b.rows)) * One(b.matrix[1][1])));
NBCheck("QGNAG_TestAdjugateBlocks", QGNAG_TestAdjugateBlocks(inv_blocks) = true);

[ OK ]  QGNAG_TestInverseBlocks
[ OK ]  inverse * matrix = identity in every block
[ OK ]  QGNAG_TestAdjugateBlocks


## 5. LaTeX files

All files are written to `OUTDIR`.

| File | Content |
|---|---|
| `quantum_characters_T4.tex` | $\epsilon$ in terms of $\nu$ |
| `the_polynomial_coefficients.tex` | the polynomials $t_{p,q}$ |
| `block_matrices.tex`, `block_matrix_form.tex` | the blocks of $M$ |
| `block_inverses.tex` | the inverses with full fractions (raw record; too wide to be `\input` in the paper) |
| `block_inverse_matrix_form.tex`, `block_adjugate_form.tex` | the inverses in matrix / adjugate form |

In [66]:
QGNAG_LoadQuantumCharactersToLaTeX(Concatenation(OUTDIR, "quantum_characters_T4.tex"), epsilon_and_coeffs, NuVectorNamesPermuted, SimpleNames);
QGNAG_ThePolynomialCoefficients(Concatenation(OUTDIR, "the_polynomial_coefficients.tex"), Permuted(data_record_data_decomp, sigma), epsilon_and_coeffs, SimpleNames);

Length(nu) = 248
Shift counts = [ 10, 7, 4, 1, 4, 7, 1, 4, 7, 10, 7, 4, 4, 1, 1, 4, 4, 1, 4, 1, 4, 7, 4, 7, 1, 4, 1, 4, 7, 4, 7, 4, 3, 6, 3, 6, 3, 6, 8, 5, 2, 5, 2, 5, 6, 3, 6, 3, 6, 3, 5, 2, 5, 8, 5, 2 ]


In [71]:
QGNAG_LoadBlockMatricesToLaTeX(Concatenation(OUTDIR, "block_matrices.tex"), blocks);
QGNAG_LoadBlockMatrixFormToLaTeX(Concatenation(OUTDIR, "block_matrix_form.tex"), blocks);
QGNAG_LoadBlockInversesToLaTeX(Concatenation(OUTDIR, "block_inverses.tex"), inv_blocks);
QGNAG_LoadBlockInverseMatrixFormToLaTeX(Concatenation(OUTDIR, "block_inverse_matrix_form.tex"), inv_blocks);
QGNAG_LoadBlockAdjugateFormToLaTeX(Concatenation(OUTDIR, "block_adjugate_form.tex"), inv_blocks, 1);

In [72]:
NBCheck("the LaTeX files exist",
        ForAll(["quantum_characters_T4.tex", "the_polynomial_coefficients.tex", "block_matrices.tex", "block_matrix_form.tex",
                "block_inverses.tex", "block_inverse_matrix_form.tex", "block_adjugate_form.tex"],
               f -> IsExistingFile(Concatenation(OUTDIR, f))));

[ OK ]  the LaTeX files exist


### 5.1 polynomialsKL and seriesKL in the format of the paper

* `polynomialsKL.tex`: $\epsilon_i=\sum_j t_{i,j}\nu_j$, block by block (principal block, blocks of size 6 and 2 side by side, blocks of size 1 in one sentence).
  Repeated coefficients are factored, $q^{-1}(\nu_{10}+\nu_{16})$, and repeated non-monomial coefficients are named $\iota_1(q),\iota_2(q),\dots$
* `seriesKL.tex`: $\nu_i=\mathfrak d^{k}\sum_j c_{i,j}\epsilon_j$ with $\mathfrak d=(1-q^{-6})^{-1}$ and $2k$ the size of the block; the $c_{i,j}$ are Laurent polynomials
  in $q^{-1}$, written as short sums with the repeated coefficients factored.

Both writers build every coefficient from its **Laurent coefficients**; no printed string is parsed.

**Warning about the older printers.** `QGNAG_LoadBlockInversesToLaTeX`, `QGNAG_LoadBlockInverseMatrixFormToLaTeX` and `QGNAG_LoadBlockAdjugateFormToLaTeX`
use `QGNAG_CoeffToLaTeX`, which (in the version before the fix of `coefftolatex_fix.g`) moves a leading minus sign out of a numerator with several terms and
negates only its first term: `(-2*q^15-q^9)` was written `-(2q^{15}-q^{9})`. This produced the wrong signs of the inversion formulas printed in the paper.

In [72]:
# polynomialsKL and seriesKL in the format of the paper (same code as kltex.gd / kltex.gi of the package QGNAG).
# The definitions are skipped if the package already provides them.
if not IsBound(QGNAG_WriteSeriesKLTeX) then
    # Terms [a, e] of a Laurent polynomial c = sum a q^e, ordered by decreasing e (q^0, q^-1, q^-2, ...).
    QGNAG_LaurentTerms := function(c)
        local cv, terms, i;
        if IsZero(c) then
            return [];
        fi;
        cv := CoefficientsOfLaurentPolynomial(c);
        terms := [];
        for i in [1 .. Length(cv[1])] do
            if not IsZero(cv[1][i]) then
                Add(terms, [cv[1][i], cv[2] + i - 1]);
            fi;
        od;
        Sort(terms, function(s, t) return s[2] > t[2]; end);
        return terms;
    end;;

    # |a| q^e without sign:  3q^{-4},  q^{-4},  3.
    QGNAG_MonomialToTeX := function(a, e)
        local absA, coef;
        absA := AbsoluteValue(a);
        if e = 0 then
            return String(absA);
        fi;
        if absA = 1 then
            coef := "";
        else
            coef := String(absA);
        fi;
        return Concatenation(coef, "q^{", String(e), "}");
    end;;

    # Laurent polynomial as a signed sum:  -2q^{-3} - q^{-9},  1 + 2q^{-6}.
    QGNAG_LaurentToTeX := function(c)
        local terms, text, k;
        terms := QGNAG_LaurentTerms(c);
        if IsEmpty(terms) then
            return "0";
        fi;
        text := "";
        for k in [1 .. Length(terms)] do
            if k = 1 then
                if terms[k][1] < 0 then
                    Append(text, "-");
                fi;
            elif terms[k][1] < 0 then
                Append(text, " - ");
            else
                Append(text, " + ");
            fi;
            Append(text, QGNAG_MonomialToTeX(terms[k][1], terms[k][2]));
        od;
        return text;
    end;;

    # [sign, text] of c * name.  A single term keeps its sign outside; a sum keeps its own signs inside parentheses.
    # iotas = list of [coefficient, "\iota_{j}(q)"].
    QGNAG_CoefficientTimes := function(c, name, iotas)
        local terms, a, e, body, i;
        terms := QGNAG_LaurentTerms(c);
        if Length(terms) = 1 then
            a := terms[1][1];
            e := terms[1][2];
            if e = 0 and AbsoluteValue(a) = 1 then
                body := name;
            else
                body := Concatenation(QGNAG_MonomialToTeX(a, e), name);
            fi;
            if a < 0 then
                return ["-", body];
            fi;
            return ["+", body];
        fi;
        for i in iotas do
            if i[1] = c then
                return ["+", Concatenation(i[2], name)];
            fi;
        od;
        return ["+", Concatenation("(", QGNAG_LaurentToTeX(c), ")", name)];
    end;;

    # Estimated width of a TeX string (number of visible glyphs): commands such as \epsilon count as one glyph,
    # \Big, \big, \quad, braces, ^, _ and spaces count as nothing.
    QGNAG_VisibleLength := function(s)
        local n, i, j, c, cmd, letters;
        letters := "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ";
        n := 0;
        i := 1;
        while i <= Length(s) do
            c := s[i];
            if c = '\\' then
                j := i + 1;
                while j <= Length(s) and s[j] in letters do
                    j := j + 1;
                od;
                cmd := s{[i + 1 .. j - 1]};
                if not cmd in ["Big", "big", "quad"] then
                    n := n + 1;
                fi;
                i := Maximum(j, i + 1);
            elif c in "{}_^ \n" then
                i := i + 1;
            else
                n := n + 1;
                i := i + 1;
            fi;
        od;
        return n;
    end;;

    # terms = list of [sign, text]. A line break (LaTeX \\) and the text cont are inserted when the estimated width of the
    # line would exceed maxWidth; offset is the width already used on the first line (left-hand side, \d^3\Big( ...).
    QGNAG_JoinSignedTerms := function(terms, maxWidth, cont, offset)
        local text, k, w, cur;
        text := "";
        cur  := offset;
        for k in [1 .. Length(terms)] do
            w := 3 + QGNAG_VisibleLength(terms[k][2]);
            if k = 1 then
                if terms[k][1] = "-" then
                    Append(text, "-");
                fi;
                Append(text, terms[k][2]);
                cur := cur + w;
            elif cur + w > maxWidth then
                Append(text, Concatenation("\\\\\n", cont, terms[k][1], " ", terms[k][2]));
                cur := w;
            else
                Append(text, Concatenation(" ", terms[k][1], " ", terms[k][2]));
                cur := cur + w;
            fi;
        od;
        return text;
    end;;

    # One equation  lhs sep wrapOpen (sum of c * symbol_j) wrapClose.  Entries with the same coefficient are factored.
    QGNAG_EquationToTeX := function(lhs, row, idx, symbol, factor, iotas, cont, maxWidth, sep, wrapOpen, wrapClose)
        local groups, g, j, found, terms, name;
        groups := [];
        for j in [1 .. Length(row)] do
            if not IsZero(row[j]) then
                found := false;
                if factor then
                    for g in groups do
                        if g[1] = row[j] then
                            Add(g[2], idx[j]);
                            found := true;
                            break;
                        fi;
                    od;
                fi;
                if not found then
                    Add(groups, [row[j], [idx[j]]]);
                fi;
            fi;
        od;
        terms := [];
        for g in groups do
            if Length(g[2]) = 1 then
                name := Concatenation(symbol, "_{", String(g[2][1]), "}");
            else
                name := Concatenation("(", JoinStringsWithSeparator(
                            List(g[2], i -> Concatenation(symbol, "_{", String(i), "}")), "+"), ")");
            fi;
            Add(terms, QGNAG_CoefficientTimes(g[1], name, iotas));
        od;
        return Concatenation(lhs, sep, wrapOpen,
                             QGNAG_JoinSignedTerms(terms, maxWidth, cont, QGNAG_VisibleLength(Concatenation(lhs, sep, wrapOpen))), wrapClose);
    end;;

    # Non-monomial coefficients that appear at least twice in a block, named \iota_{1}(q), \iota_{2}(q), ...
    QGNAG_IotasOfBlock := function(matrix)
        local found, row, c, t, i, iotas, k;
        found := [];
        for row in matrix do
            for c in row do
                if Length(QGNAG_LaurentTerms(c)) > 1 then
                    t := false;
                    for i in found do
                        if i[1] = c then
                            i[2] := i[2] + 1;
                            t := true;
                        fi;
                    od;
                    if not t then
                        Add(found, [c, 1]);
                    fi;
                fi;
            od;
        od;
        iotas := [];
        k := 0;
        for i in found do
            if i[2] >= 2 then
                k := k + 1;
                Add(iotas, [i[1], Concatenation("\\iota_{", String(k), "}(q)")]);
            fi;
        od;
        return iotas;
    end;;

    # Smallest k such that  d^{-k} * inverse  has Laurent polynomial entries, with d = (1-q^-6)^-1.
    QGNAG_DegreeOfD := function(b)
        local q, d6, k;
        q  := IndeterminateOfUnivariateRationalFunction(b.det);
        d6 := One(q) - q^-6;
        k  := 0;
        while not ForAll(b.inverse, r -> ForAll(r, c -> IsLaurentPolynomial(c * d6^k))) do
            k := k + 1;
            if k > Length(b.rows) + 2 then
                Error("QGNAG_DegreeOfD: the inverse of the block ", b.rows, " is not d^k times a Laurent polynomial");
            fi;
        od;
        return k;
    end;;

    # The equations of one block (list of strings). mode = "align" or "cases".
    QGNAG_BlockEquations := function(b, kind, factor, iotas, mode, maxWidth)
        local eqs, r, lhs, sep, cont, k, q, d6, row, wo, wc;
        eqs := [];
        if mode = "align" then
            cont := "&";
        else
            cont := "\\quad";
        fi;
        if kind = "polynomials" then
            if mode = "align" then sep := " &= "; else sep := " = "; fi;
            for r in [1 .. Length(b.rows)] do
                lhs := Concatenation("\\epsilon_{", String(b.rows[r]), "}");
                Add(eqs, QGNAG_EquationToTeX(lhs, b.matrix[r], b.cols, "\\nu", factor, iotas, cont, maxWidth, sep, "", ""));
            od;
        else
            if mode = "align" then sep := " =& "; else sep := " = "; fi;
            k  := QGNAG_DegreeOfD(b);
            q  := IndeterminateOfUnivariateRationalFunction(b.det);
            d6 := One(q) - q^-6;
            if k >= 3 then
                wo := Concatenation("\\d^{", String(k), "}\\Big(");  wc := "\\Big)";
            elif k = 2 then
                wo := "\\d^{2}\\big(";                                wc := "\\big)";
            else
                wo := "\\d(";                                         wc := ")";
            fi;
            for r in [1 .. Length(b.cols)] do
                row := List(b.inverse[r], c -> c * d6^k);
                lhs := Concatenation("\\nu_{", String(b.cols[r]), "}");
                Add(eqs, QGNAG_EquationToTeX(lhs, row, b.rows, "\\epsilon", factor, [], cont, maxWidth, sep, wo, wc));
            od;
        fi;
        return eqs;
    end;;

    QGNAG_KLTeXText := function(blocks, kind, factor, widths)
        local text, sizes, size, group, b, eqs, items, k, i, perRow, iotas, heading, ones, defs, j, r, row, nu, ep, mode, rowsText;
        text  := "";
        sizes := Reversed(Set(List(blocks, b -> Length(b.rows))));
        for size in sizes do
            group := Filtered(blocks, b -> Length(b.rows) = size);
            if size = 1 then
                ones := List(group, b -> b.rows[1]);
                if kind = "polynomials" then
                    nu := "\\epsilon_i=\\nu_i";
                else
                    nu := "\\nu_i=\\epsilon_i";
                fi;
                text := Concatenation(text, "\\subsubsection{The blocks of size 1}\n",
                        "For $i\\in\\{", JoinStringsWithSeparator(List(ones, String), ", "), "\\}$ we have $", nu, "$.\n\n");
            elif size > 6 then
                for b in group do
                    if kind = "polynomials" and 1 in b.rows then
                        heading := "The principal block";
                    elif Length(group) = 1 then
                        heading := Concatenation("The block of size ", String(size));
                    else
                        heading := Concatenation("The blocks of size ", String(size));
                    fi;
                    iotas := [];
                    if kind = "polynomials" then
                        iotas := QGNAG_IotasOfBlock(b.matrix);
                    fi;
                    eqs := QGNAG_BlockEquations(b, kind, factor, iotas, "align", widths[1]);
                    text := Concatenation(text, "\\subsubsection{", heading, "}\n\n\\begin{align*}\n");
                    for i in [1 .. Length(eqs)] do
                        text := Concatenation(text, eqs[i]);
                        if i < Length(eqs) then
                            text := Concatenation(text, ",\\\\\n");
                        else
                            text := Concatenation(text, ".\n");
                        fi;
                    od;
                    text := Concatenation(text, "\\end{align*}\n");
                    if Length(iotas) > 0 then
                        defs := List(iotas, i -> Concatenation("$", i[2], "\\coloneqq ", QGNAG_LaurentToTeX(i[1]), "$"));
                        text := Concatenation(text, "Here ", JoinStringsWithSeparator(defs, ", "), ".\n");
                    fi;
                    text := Concatenation(text, "\n");
                od;
            else
                if size = 6 then
                    if kind = "polynomials" then perRow := 2; else perRow := 1; fi;
                elif kind = "polynomials" then
                    perRow := 3;
                else
                    perRow := 2;
                fi;
                if Length(group) = 1 then
                    heading := Concatenation("The block of size ", String(size));
                else
                    heading := Concatenation("The blocks of size ", String(size));
                fi;
                items := [];
                for b in group do
                    if perRow = 1 then
                        eqs := QGNAG_BlockEquations(b, kind, factor, [], "cases", widths[3]);
                    else
                        eqs := QGNAG_BlockEquations(b, kind, factor, [], "cases", widths[2]);
                    fi;
                    rowsText := "";
                    for i in [1 .. Length(eqs)] do
                        rowsText := Concatenation(rowsText, eqs[i]);
                        if i < Length(eqs) then
                            rowsText := Concatenation(rowsText, ",\\\\\n");
                        else
                            rowsText := Concatenation(rowsText, ".\n");
                        fi;
                    od;
                    Add(items, Concatenation("\\begin{cases}\n", rowsText, "\\end{cases}"));
                od;
                text := Concatenation(text, "\\subsubsection{", heading, "}\n\n\\begin{align*}\n");
                k := 0;
                while k < Length(items) do
                    row := items{[k + 1 .. Minimum(k + perRow, Length(items))]};
                    text := Concatenation(text, "&", JoinStringsWithSeparator(row, "\n&&"));
                    k := k + perRow;
                    if k < Length(items) then
                        text := Concatenation(text, "\n\\\\\n");
                    else
                        text := Concatenation(text, "\n");
                    fi;
                od;
                text := Concatenation(text, "\\end{align*}\n\n");
            fi;
        od;
        return text;
    end;;

    QGNAG_WritePolynomialsKLTeX := function(filename, blocks)
        local out;
        out := OutputTextFile(filename, false);
        SetPrintFormattingStatus(out, false);
        AppendTo(out, "% Verma characters in terms of the irreducible characters, generated by QGNAG_WritePolynomialsKLTeX\n",
                      "% Requires: amsmath, \\coloneqq (mathtools)\n\n",
                      QGNAG_KLTeXText(blocks, "polynomials", true, [ 70, 29, 60 ]));
        CloseStream(out);
        return true;
    end;;

    QGNAG_WriteSeriesKLTeX := function(filename, inv_blocks)
        local out;
        out := OutputTextFile(filename, false);
        SetPrintFormattingStatus(out, false);
        AppendTo(out, "% Irreducible characters in terms of the Verma characters, generated by QGNAG_WriteSeriesKLTeX\n",
                      "% Requires: amsmath;  \\d = (1-q^{-6})^{-1}\n\n",
                      QGNAG_KLTeXText(inv_blocks, "series", true, [ 70, 29, 60 ]));
        CloseStream(out);
        return true;
    end;;

fi;

In [75]:
QGNAG_WritePolynomialsKLTeX(Concatenation(OUTDIR, "polynomialsKL.tex"), blocks);
QGNAG_WriteSeriesKLTeX(Concatenation(OUTDIR, "seriesKL.tex"), inv_blocks);

true

true

In [79]:
# Checks against the review (Appendix B) and the matrix of quantum polynomials.
NBCheck("polynomialsKL.tex: eps_1 = nu_1 + q^-9 nu_10 + q^-1 nu_36 + q^-4 nu_45",
        NBContains(Concatenation(OUTDIR, "polynomialsKL.tex"), "\\epsilon_{1} &= \\nu_{1} + q^{-9}\\nu_{10} + q^{-1}\\nu_{36} + q^{-4}\\nu_{45}"));
NBCheck("seriesKL.tex: the coefficient of eps_10 in nu_1 is (-2q^{-3} - q^{-9})  (review, Appendix B)",
        NBContains(Concatenation(OUTDIR, "seriesKL.tex"), "(-2q^{-3} - q^{-9})\\epsilon_{10}"));
NBCheck("seriesKL.tex: the coefficient of eps_36 in nu_1 is (-q^{-1} - 2q^{-7})  (review, Appendix B)",
        NBContains(Concatenation(OUTDIR, "seriesKL.tex"), "(-q^{-1} - 2q^{-7})\\epsilon_{36}"));
# Informational: which version of QGNAG_CoeffToLaTeX wrote block_inverses.tex?
if NBContains(Concatenation(OUTDIR, "block_inverses.tex"), "\\frac{2q^{15}-q^{9}}{") then
    Print("[WARN]  block_inverses.tex was written with the OLD QGNAG_CoeffToLaTeX (wrong signs): apply coefftolatex_fix.g\n");
elif NBContains(Concatenation(OUTDIR, "block_inverses.tex"), "\\frac{-2q^{15}-q^{9}}{") then
    Print("[INFO]  block_inverses.tex was written with the fixed QGNAG_CoeffToLaTeX\n");
fi;

[ OK ]  polynomialsKL.tex: eps_1 = nu_1 + q^-9 nu_10 + q^-1 nu_36 + q^-4 nu_45
[ OK ]  seriesKL.tex: the coefficient of eps_10 in nu_1 is (-2q^{-3} - q^{-9})  (review, Appendix B)
[ OK ]  seriesKL.tex: the coefficient of eps_36 in nu_1 is (-q^{-1} - 2q^{-7})  (review, Appendix B)
[INFO]  block_inverses.tex was written with the fixed QGNAG_CoeffToLaTeX


## 6. Characters (format of the paper)

`QGNAG_WriteCharactersTeX(data_record_data_decomp, SimpleNames, filename, filtered)` writes, for each simple $L(\lambda)$, the line
`\car(\lambda) = ...` in the format of the paper: one `align*` per family ($\tau,\rho,\chi,\eta$), the terms ordered by increasing degree
and the repeated multiplicities factored, e.g. `2\cdot(\underline{8}+\underline{34})q^{-3}`.

* `characters.tex` (`filtered = true`): the **46** weights printed in the paper. The weights whose $L(\lambda)$ is a simple Verma module
  (decomposition in all the degrees) and those of dimension 1 (a single degree) are skipped.
* `characters_all.tex` (`filtered = false`): the 56 weights.

The file is meant to be `\input` in the paper; its preamble must define `\car` (= `\operatorname{char}`) and use `\allowdisplaybreaks`.

In [79]:
# Characters in the format of the paper.  The same code is in characters.gd / characters.gi (package QGNAG);
# the definitions are skipped if the package already provides them.
if not IsBound(QGNAG_WriteCharactersTeX) then
    # Term of one degree.  pairs = [ [mult, idx], ... ] at degree d (d <= 0).
    #   one multiplicity, 1 :  \underline{a}\,q^{d}            or   (\underline{a}+\underline{b})q^{d}
    #   one multiplicity, m :  m\cdot\underline{a}\,q^{d}      or   m\cdot(\underline{a}+\underline{b})q^{d}
    #   several             :  (\underline{a}+\underline{b}+m\cdot(\underline{c}+\underline{d}))q^{d}
    # (the multiplicity-1 terms first; then the groups by increasing multiplicity).
    QGNAG_CharacterDegreeTermToTeX := function(pairs, d, style)
        local u, mults, parts, m, idxs, body, thin, qpart;

        u := i -> Concatenation("\\underline{", String(i), "}");

        mults := Set(List(pairs, p -> p[1]));
        parts := [];
        for m in mults do
            idxs := List(Filtered(pairs, p -> p[1] = m), p -> p[2]);
            Sort(idxs);
            if m = 1 then
                Add(parts, JoinStringsWithSeparator(List(idxs, u), "+"));
            elif Length(idxs) = 1 then
                Add(parts, Concatenation(String(m), "\\cdot", u(idxs[1])));
            else
                Add(parts, Concatenation(String(m), "\\cdot(",
                                         JoinStringsWithSeparator(List(idxs, u), "+"), ")"));
            fi;
        od;

        thin := false;
        if Length(mults) = 1 then
            if mults[1] = 1 and Length(pairs) > 1 then
                body := Concatenation("(", parts[1], ")");
            else
                body := parts[1];
                thin := (Length(pairs) = 1);          # a single underline: add a thin space before q
            fi;
        else
            body := Concatenation("(", JoinStringsWithSeparator(parts, "+"), ")");
        fi;

        if style = "bracket" then
            return [ Concatenation(body, "[", String(d), "]"), Length(pairs) ];
        fi;
        if d = 0 then
            qpart := "q^0";
        else
            qpart := Concatenation("q^{", String(d), "}");
        fi;
        if thin then
            qpart := Concatenation("\\,", qpart);
        fi;
        return [ Concatenation(body, qpart), Length(pairs) ];
    end;;

    # One character:  \car(name) &= term + term + ...   with a LaTeX line break when a line would have
    # more than maxItems underlines.
    QGNAG_CharacterToTeX := function(rec_info, LName, maxItems, style)
        local degrees, d, t, text, count, first;

        degrees := List(RecNames(rec_info), Int);
        Sort(degrees);
        text  := Concatenation("\\car", LName, "\n\t&=");
        count := 0;
        first := true;
        for d in degrees do
            t := QGNAG_CharacterDegreeTermToTeX(rec_info.(String(d)), d, style);
            if not first then
                if count + t[2] > maxItems then
                    text  := Concatenation(text, "\\\\\n\t&\\quad+");
                    count := 0;
                else
                    text := Concatenation(text, "+");
                fi;
            fi;
            text  := Concatenation(text, t[1]);
            count := count + t[2];
            first := false;
        od;
        return text;
    end;;

    QGNAG_CharactersTeXText := function(permuted_decomps, sNamesL, n_top, filtered, maxItems, style)
        local families, fam, kept, i, nrec, text, k, skip;

        families := [ [ "\\tau",  "\\Lambda_\\tau'"  ],
                      [ "\\rho",  "\\Lambda_\\rho'"  ],
                      [ "\\chi",  "\\Lambda_\\chi'"  ],
                      [ "\\eta",  "\\Lambda_\\eta'"  ] ];
        text := "";
        for fam in families do
            kept := [];
            for i in [1 .. Length(permuted_decomps)] do
                nrec := Length(RecNames(permuted_decomps[i]));
                skip := filtered and (nrec = n_top or nrec = 1);
                if not skip and PositionSublist(sNamesL[i], fam[1]) <> fail then
                    Add(kept, i);
                fi;
            od;
            if Length(kept) > 0 then
                text := Concatenation(text, "\\subsubsection{Case $\\lambda\\in ", fam[2], "$}\n\n\\begin{align*}\n");
                for k in [1 .. Length(kept)] do
                    i    := kept[k];
                    text := Concatenation(text, "\t",
                                          QGNAG_CharacterToTeX(permuted_decomps[i], sNamesL[i], maxItems, style));
                    if k < Length(kept) then
                        text := Concatenation(text, ".\\\\\n");
                    else
                        text := Concatenation(text, ".\n");
                    fi;
                od;
                text := Concatenation(text, "\\end{align*}\n\n");
            fi;
        od;
        return text;
    end;;

    QGNAG_WriteCharactersTeX := function(data_record_data_decomp, SimpleNames, filename, filtered)
        local tau, data_neg, data_neg_permuted, sNamesL, n_top, text, out;

        tau               := QGNAG_PermutationVarChars(data_record_data_decomp);
        data_neg          := List(data_record_data_decomp, QGNAG_PrepareDecomposition);
        data_neg_permuted := Permuted(data_neg, tau);
        sNamesL           := List(SimpleNames, QGNAG_ExtractTuple);
        n_top             := Length(RecNames(data_record_data_decomp[1][1]));

        text := Concatenation(
            "% Characters of the simple modules L(lambda), generated by QGNAG_WriteCharactersTeX\n",
            "% Requires: amsmath;  \\car = \\operatorname{char}\n\n",
            QGNAG_CharactersTeXText(data_neg_permuted, sNamesL, n_top, filtered, 7, "q"));

        out := OutputTextFile(filename, false);
        SetPrintFormattingStatus(out, false);
        AppendTo(out, text);
        CloseStream(out);
        return true;
    end;;

fi;

In [82]:
QGNAG_WriteCharactersTeX(data_record_data_decomp, SimpleNames, Concatenation(OUTDIR, "characters.tex"),     true);
QGNAG_WriteCharactersTeX(data_record_data_decomp, SimpleNames, Concatenation(OUTDIR, "characters_all.tex"), false);

true

true

In [85]:
# Number of characters (\car) written in a file.
NBCountCar := function(filename)
    local s, i, n;
    s := StringFile(filename);
    n := 0;
    for i in [1 .. Length(s) - 4] do
        if s{[i .. i + 4]} = "\\car(" then
            n := n + 1;
        fi;
    od;
    return n;
end;;
NBCheck("characters.tex: 46 characters (those printed in the paper)", NBCountCar(Concatenation(OUTDIR, "characters.tex")) = 46);
NBCheck("characters_all.tex: 56 characters",                           NBCountCar(Concatenation(OUTDIR, "characters_all.tex")) = 56);

[ OK ]  characters.tex: 46 characters (those printed in the paper)
[ OK ]  characters_all.tex: 56 characters


Optional: characters, Hilbert series and dimensions of all the simples (function of the package; it needs `SimplesMn`).

In [85]:
if IsBound(SimplesMn) and IsBoundGlobal("QGNAG_WriteHilbertSeriesTeX") then
    QGNAG_WriteHilbertSeriesTeX(data_record_data_decomp, SimpleNames, SimplesMn, Concatenation(OUTDIR, "char_and_hilbert.tex"));
else
    Print("[SKIP]  char_and_hilbert.tex: SimplesMn or QGNAG_WriteHilbertSeriesTeX is not available\n");
fi;

[SKIP]  char_and_hilbert.tex: SimplesMn or QGNAG_WriteHilbertSeriesTeX is not available


## 7. Summary of the checks

In [86]:
Print("\n", Number(NB_RESULTS, x -> x), " of ", Length(NB_RESULTS), " checks passed.\n");
if ForAll(NB_RESULTS, x -> x) then
    Print("ALL CHECKS PASSED\n");
else
    Print("SOME CHECKS FAILED: see the lines starting with [FAIL]\n");
fi;


34 of 34 checks passed.
ALL CHECKS PASSED


## Annex A. Functions under development (defined only; **not yet validated**)

They were copied unchanged from the previous notebook, where they had not been executed. Running these cells only *defines* the functions
(a definition is skipped if the package already provides it). They expose the Hilbert-series data of `QGNAG_WriteHilbertSeriesTeX` as lists
and test the palindromicity of the series.

In [88]:
if not IsBound(QGNAG_HilbertSeriesCoefficients) then
#############################################################################
##
##  Utility functions closely related to QGNAG_WriteHilbertSeriesTeX:
##  they expose the same per-weight Hilbert series computation as plain
##  data (lists of [coefficient, degree] pairs) instead of LaTeX text,
##  and check whether that data is palindromic.
##
##  All functions below reuse the same tau-permutation as
##  QGNAG_WriteHilbertSeriesTeX, so result[i] here corresponds to the
##  same i-th simple (same sNamesL[i]) as in that function.
##
#############################################################################


#############################################################################
##
#F  QGNAG_HilbertSeriesCoefficients( <data_record_data_decomp>, <degreeofreps> )
##
##  For each of the theta simple modules, computes its Hilbert series as a
##  list of [coefficient, degree] pairs, sorted by increasing degree
##  (this is exactly the "total" computed inside the Hilbert-series block
##  of QGNAG_WriteHilbertSeriesTeX, returned as data instead of text).
##
##  Returns a list of length theta; entry i looks e.g. like
##  [ [1,-3], [2,-2], [3,-1], [1,0] ].
##
QGNAG_HilbertSeriesCoefficients := function(data_record_data_decomp, degreeofreps)
    local tau,
          data_rec_info_to_zero_negative,
          data_rec_info_to_zero_negative_permuted,
          theta,
          i,
          rec_info,
          degrees,
          d,
          pair,
          mult,
          idx,
          total,
          series,
          result;

    tau := QGNAG_PermutationVarChars(data_record_data_decomp);
    data_rec_info_to_zero_negative := List(data_record_data_decomp, QGNAG_PrepareDecomposition);
    data_rec_info_to_zero_negative_permuted := Permuted(data_rec_info_to_zero_negative, tau);
    theta  := Length(data_rec_info_to_zero_negative_permuted);
    result := [];

    for i in [1 .. theta] do
        rec_info := data_rec_info_to_zero_negative_permuted[i];
        degrees := List(RecNames(rec_info), Int);
        Sort(degrees);
        series := [];
        for d in degrees do
            total := 0;
            for pair in rec_info.(String(d)) do
                mult  := pair[1];
                idx   := pair[2];
                total := total + mult * degreeofreps[idx];
            od;
            Add(series, [total, d]);
        od;
        Add(result, series);
    od;
    return result;
end;
fi;

In [89]:
if not IsBound(QGNAG_HilbertSeriesNegativePart) then
#############################################################################
##
#F  QGNAG_HilbertSeriesNegativePart( <data_record_data_decomp>, <degreeofreps> )
##
##  Same data as QGNAG_HilbertSeriesCoefficients, but keeps, for each
##  simple, only the pairs [coefficient, degree] with degree <= 0 (the
##  "negative grading" half of the series, including the degree-0 term,
##  as in [[2,-2],[6,-1],[3,0]]).
##
##  If you actually want strictly negative degrees (excluding 0), change
##  "pair[2] <= 0" below to "pair[2] < 0".
##
##  Returns a list of length theta.
##
QGNAG_HilbertSeriesNegativePart := function(data_record_data_decomp, degreeofreps)
    local full, result, series, neg_part;

    full   := QGNAG_HilbertSeriesCoefficients(data_record_data_decomp, degreeofreps);
    result := [];

    for series in full do
        neg_part := Filtered(series, pair -> pair[2] <= 0);
        Add(result, neg_part);
    od;

    return result;
end;
fi;

In [90]:
if not IsBound(QGNAG_IsPal) then
#############################################################################
##
#F  QGNAG_IsPal( <L> )
##
##  Direct translation of the board's "IsPal" algorithm: L is a list of
##  pairs [coefficient, degree] (or any list of lists), sorted by
##  increasing degree. Checks whether the sequence of first components
##  (coefficients) reads the same forwards and backwards:
##  for k = 1, ..., Floor(Length(L)/2), compares L[k][1] with
##  L[Length(L)-k+1][1]; returns false at the first mismatch, true
##  otherwise.
##
##  Note: as in the board, only the coefficients (position [1]) are
##  compared, not the degrees themselves.
##
QGNAG_IsPal := function(L)
    local m, k;

    m := Length(L);

    for k in [1 .. QuoInt(m, 2)] do
        if L[k][1] <> L[m - k + 1][1] then
            return false;
        fi;
    od;

    return true;
end;
fi;

In [91]:
if not IsBound(QGNAG_HilbertSeriesIsPalindromic) then
#############################################################################
##
#F  QGNAG_HilbertSeriesIsPalindromic( <data_record_data_decomp>, <degreeofreps> )
##
##  For each of the theta simple modules, computes its full Hilbert
##  series (QGNAG_HilbertSeriesCoefficients) and checks whether it is
##  palindromic (QGNAG_IsPal). The full series is used on purpose: to
##  check palindromy you need both halves, not just the negative part.
##
##  Returns a list of length theta of true/false values, in the same
##  order as QGNAG_HilbertSeriesCoefficients (and as sNamesL in
##  QGNAG_WriteHilbertSeriesTeX).
##
QGNAG_HilbertSeriesIsPalindromic := function(data_record_data_decomp, degreeofreps)
    local full, result, series;

    full   := QGNAG_HilbertSeriesCoefficients(data_record_data_decomp, degreeofreps);
    result := [];

    for series in full do
        Add(result, QGNAG_IsPal(series));
    od;

    return result;
end;
fi;

## Annex B. Optional console views (long output)

In [93]:
QGNAG_PrintBlockCombinations(blocks);

Block 1  (12 \times 12)   epsilons = [ 1, 10, 13, 16, 17, 19, 26, 28, 36, 39, 45, 54 ]   nus = [ 1, 10, 13, 16, 17, 19, 26, 28, 36, 39, 45, 54 ]
  \epsilon_{1} = \nu_{1} + q^-9*\nu_{10} + q^-1*\nu_{36} + q^-4*\nu_{45}
  \epsilon_{10} = q^-9*\nu_{1} + \nu_{10} + q^-4*\nu_{36} + q^-1*\nu_{45}
  \epsilon_{13} = q^-6*\nu_{1} + q^-3*\nu_{10} + \nu_{13} + q^-3*\nu_{16} + q^-1*\nu_{36} + q^-2*\nu_{39} + q^-4*\nu_{45} + q^-5*\nu_{54}
  \epsilon_{16} = q^-3*\nu_{1} + q^-6*\nu_{10} + q^-3*\nu_{13} + \nu_{16} + q^-4*\nu_{36} + q^-5*\nu_{39} + q^-1*\nu_{45} + q^-2*\nu_{54}
  \epsilon_{17} = q^-6*\nu_{1} + q^-3*\nu_{10} + \nu_{17} + q^-3*\nu_{26} + q^-1*\nu_{36} + q^-2*\nu_{39} + q^-4*\nu_{45} + q^-5*\nu_{54}
  \epsilon_{19} = q^-6*\nu_{1} + q^-3*\nu_{10} + \nu_{19} + q^-3*\nu_{28} + q^-1*\nu_{36} + q^-2*\nu_{39} + q^-4*\nu_{45} + q^-5*\nu_{54}
  \epsilon_{26} = q^-3*\nu_{1} + q^-6*\nu_{10} + q^-3*\nu_{17} + \nu_{26} + q^-4*\nu_{36} + q^-5*\nu_{39} + q^-1*\nu_{45} + q^-2*\nu_{54}
  \epsilon_{28} = 

true

In [94]:
QGNAG_PrintInverseCombinations(inv_blocks);   # nu_k as a combination of eps

Block 1  (12 x 12)   det = 1-10*q^-6+45*q^-12-120*q^-18+210*q^-24-252*q^-30+210*q^-36-120*q^-42+45*q^-48-10*q^-54+q^-60
  nu_1 = ((q^18+2*q^12)/(q^18-3*q^12+3*q^6-1))*eps_1 + ((-2*q^15-q^9)/(q^18-3*q^12+3*q^6-1))*eps_10 + ((q^12)/(q^18-3*q^12+3*q^6-1))*eps_13 + ((-q^15)/(q^18-3*q^12+3*q^6-1))*eps_16 + ((q^12)/(q^18-3*q^12+3*q^6-1))*eps_17 + ((q^12)/(q^18-3*q^12+3*q^6-1))*eps_19 + ((-q^15)/(q^18-3*q^12+3*q^6-1))*eps_26 + ((-q^15)/(q^18-3*q^12+3*q^6-1))*eps_28 + ((-q^17-2*q^11)/(q^18-3*q^12+3*q^6-1))*eps_36 + ((2*q^16+q^10)/(q^18-3*q^12+3*q^6-1))*eps_39 + ((3*q^14)/(q^18-3*q^12+3*q^6-1))*eps_45 + ((-3*q^13)/(q^18-3*q^12+3*q^6-1))*eps_54
  nu_10 = ((-2*q^15-q^9)/(q^18-3*q^12+3*q^6-1))*eps_1 + ((q^18+2*q^12)/(q^18-3*q^12+3*q^6-1))*eps_10 + ((-q^15)/(q^18-3*q^12+3*q^6-1))*eps_13 + ((q^12)/(q^18-3*q^12+3*q^6-1))*eps_16 + ((-q^15)/(q^18-3*q^12+3*q^6-1))*eps_17 + ((-q^15)/(q^18-3*q^12+3*q^6-1))*eps_19 + ((q^12)/(q^18-3*q^12+3*q^6-1))*eps_26 + ((q^12)/(q^18-3*q^12+3*q^6-1))*eps_28 + ((3*q^14)/(

true